In [1]:
import os
import gc
import timeit
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.sparse import csr_matrix
from sklearn.metrics import accuracy_score, recall_score

In [2]:
MODELS_DIR = pathlib.Path("../models")
RESULTS_DIR = pathlib.Path("../results")
DATA_PATH = pathlib.Path("../data/fdia_dataset_processed.npz")

RESULTS_DIR.mkdir(exist_ok=True)

data = np.load(DATA_PATH)

X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"]

X_test_lstm = np.transpose(X_test, (0, 2, 1))

print("X_test:", X_test.shape)
print("X_test LSTM:", X_test_lstm.shape)
print("y_test:", y_test.shape)

X_test: (9720, 6, 83)
X_test LSTM: (9720, 83, 6)
y_test: (9720,)


In [3]:
np.random.seed(42)

sample_size = int(0.10 * len(X_test_lstm))

timing_idx = np.random.choice(
    len(X_test_lstm),
    size=sample_size,
    replace=False
)

X_lstm_timing = X_test_lstm[timing_idx]

print("Timing fraction: 10%")
print("Timing samples:", len(X_lstm_timing))
print("Timing shape:", X_lstm_timing.shape)

Timing fraction: 10%
Timing samples: 972
Timing shape: (972, 83, 6)


In [4]:
model_path = MODELS_DIR / "LSTM_WeightPruned.keras"

model = tf.keras.models.load_model(model_path)

print("Model:", model_path.name)
print()
model.summary()

print("\nLSTM layer information:")

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.LSTM):
        weights = layer.get_weights()

        kernel = weights[0]
        recurrent_kernel = weights[1]

        kernel_sparsity = np.mean(kernel == 0) * 100
        recurrent_sparsity = np.mean(recurrent_kernel == 0) * 100

        print(f"\nLayer: {layer.name}")
        print("Units:", layer.units)
        print("Return sequences:", layer.return_sequences)
        print("Kernel shape:", kernel.shape)
        print("Recurrent kernel shape:", recurrent_kernel.shape)
        print(f"Kernel sparsity: {kernel_sparsity:.2f}%")
        print(f"Recurrent sparsity: {recurrent_sparsity:.2f}%")

Model: LSTM_WeightPruned.keras



c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 48 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 83, 75)         │        24,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 83, 75)         │        45,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        70,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         6,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 478,745 (1.83 MB)

 Trainable params: 159,581 (623.36 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 319,164 (1.22 MB)


LSTM layer information:

Layer: lstm
Units: 75
Return sequences: True
Kernel shape: (6, 300)
Recurrent kernel shape: (75, 300)
Kernel sparsity: 65.00%
Recurrent sparsity: 64.99%

Layer: lstm_1
Units: 75
Return sequences: True
Kernel shape: (75, 300)
Recurrent kernel shape: (75, 300)
Kernel sparsity: 64.99%
Recurrent sparsity: 64.99%

Layer: lstm_2
Units: 100
Return sequences: False
Kernel shape: (75, 400)
Recurrent kernel shape: (100, 400)
Kernel sparsity: 64.99%
Recurrent sparsity: 64.99%


In [5]:
print("Complete model structure:\n")

for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}")
    print(f"Type: {type(layer).__name__}")
    
    if hasattr(layer, "units"):
        print(f"Units: {layer.units}")
    
    if hasattr(layer, "activation"):
        print(f"Activation: {layer.activation.__name__}")
    
    if isinstance(layer, tf.keras.layers.LSTM):
        print(f"Return sequences: {layer.return_sequences}")
    
    weights = layer.get_weights()
    
    if weights:
        print("Weight shapes:")
        for j, weight in enumerate(weights):
            print(f"  Weight {j}: {weight.shape}")
    else:
        print("No weights")
    
    print()

Complete model structure:

Layer 0: lstm
Type: LSTM
Units: 75
Activation: tanh
Return sequences: True
Weight shapes:
  Weight 0: (6, 300)
  Weight 1: (75, 300)
  Weight 2: (300,)

Layer 1: lstm_1
Type: LSTM
Units: 75
Activation: tanh
Return sequences: True
Weight shapes:
  Weight 0: (75, 300)
  Weight 1: (75, 300)
  Weight 2: (300,)

Layer 2: lstm_2
Type: LSTM
Units: 100
Activation: tanh
Return sequences: False
Weight shapes:
  Weight 0: (75, 400)
  Weight 1: (100, 400)
  Weight 2: (400,)

Layer 3: dense
Type: Dense
Units: 64
Activation: relu
Weight shapes:
  Weight 0: (100, 64)
  Weight 1: (64,)

Layer 4: dense_1
Type: Dense
Units: 64
Activation: relu
Weight shapes:
  Weight 0: (64, 64)
  Weight 1: (64,)

Layer 5: dense_2
Type: Dense
Units: 32
Activation: relu
Weight shapes:
  Weight 0: (64, 32)
  Weight 1: (32,)

Layer 6: dense_3
Type: Dense
Units: 128
Activation: relu
Weight shapes:
  Weight 0: (32, 128)
  Weight 1: (128,)

Layer 7: dense_4
Type: Dense
Units: 16
Activation: relu
Wei

In [6]:
def sigmoid(x):
    x = np.clip(x, -88.0, 88.0)
    return 1.0 / (1.0 + np.exp(-x))


def relu(x):
    return np.maximum(x, 0.0)

In [7]:
def prepare_lstm_weights(model):
    dense_lstm_layers = []
    csr_lstm_layers = []
    dense_layers = []

    for layer in model.layers:

        if isinstance(layer, tf.keras.layers.LSTM):
            kernel, recurrent_kernel, bias = layer.get_weights()

            dense_lstm_layers.append({
                "name": layer.name,
                "kernel": kernel.astype(np.float32),
                "recurrent_kernel": recurrent_kernel.astype(np.float32),
                "bias": bias.astype(np.float32),
                "units": layer.units,
                "return_sequences": layer.return_sequences
            })

            csr_lstm_layers.append({
                "name": layer.name,
                "kernel": csr_matrix(kernel),
                "recurrent_kernel": csr_matrix(recurrent_kernel),
                "bias": bias.astype(np.float32),
                "units": layer.units,
                "return_sequences": layer.return_sequences
            })

        elif isinstance(layer, tf.keras.layers.Dense):
            kernel, bias = layer.get_weights()

            dense_layers.append({
                "name": layer.name,
                "kernel": kernel.astype(np.float32),
                "bias": bias.astype(np.float32),
                "activation": layer.activation.__name__
            })

    return dense_lstm_layers, csr_lstm_layers, dense_layers

In [8]:
dense_lstm_layers, csr_lstm_layers, dense_layers = prepare_lstm_weights(model)

print("Dense LSTM layers:", len(dense_lstm_layers))
print("CSR LSTM layers:", len(csr_lstm_layers))
print("Dense classification layers:", len(dense_layers))

Dense LSTM layers: 3
CSR LSTM layers: 3
Dense classification layers: 7


In [9]:
def manual_lstm_dense(input_data, lstm_layers, dense_layers):

    layer_output = np.asarray(input_data, dtype=np.float32)

    for layer in lstm_layers:

        kernel = layer["kernel"]
        recurrent_kernel = layer["recurrent_kernel"]
        bias = layer["bias"]

        units = layer["units"]
        return_sequences = layer["return_sequences"]

        batch_size = layer_output.shape[0]
        timesteps = layer_output.shape[1]

        hidden_state = np.zeros(
            (batch_size, units),
            dtype=np.float32
        )

        cell_state = np.zeros(
            (batch_size, units),
            dtype=np.float32
        )

        sequence_output = np.empty(
            (batch_size, timesteps, units),
            dtype=np.float32
        )

        for t in range(timesteps):

            current_input = layer_output[:, t, :]

            z = (
                current_input @ kernel
                + hidden_state @ recurrent_kernel
                + bias
            )

            z_i, z_f, z_c, z_o = np.split(z, 4, axis=1)

            input_gate = sigmoid(z_i)
            forget_gate = sigmoid(z_f)
            cell_candidate = np.tanh(z_c)
            output_gate = sigmoid(z_o)

            cell_state = (
                forget_gate * cell_state
                + input_gate * cell_candidate
            )

            hidden_state = output_gate * np.tanh(cell_state)

            sequence_output[:, t, :] = hidden_state

        if return_sequences:
            layer_output = sequence_output
        else:
            layer_output = hidden_state

    for layer in dense_layers:

        layer_output = (
            layer_output @ layer["kernel"]
            + layer["bias"]
        )

        if layer["activation"] == "relu":
            layer_output = relu(layer_output)

        elif layer["activation"] == "sigmoid":
            layer_output = sigmoid(layer_output)

    return layer_output

In [12]:
def manual_lstm_csr(input_data, lstm_layers, dense_layers):

    layer_output = np.asarray(input_data, dtype=np.float32)

    for layer in lstm_layers:

        kernel = layer["kernel"]
        recurrent_kernel = layer["recurrent_kernel"]
        bias = layer["bias"]

        units = layer["units"]
        return_sequences = layer["return_sequences"]

        batch_size = layer_output.shape[0]
        timesteps = layer_output.shape[1]

        hidden_state = np.zeros(
            (batch_size, units),
            dtype=np.float32
        )

        cell_state = np.zeros(
            (batch_size, units),
            dtype=np.float32
        )

        sequence_output = np.empty(
            (batch_size, timesteps, units),
            dtype=np.float32
        )

        for t in range(timesteps):

            current_input = layer_output[:, t, :]

            input_part = kernel.T.dot(current_input.T).T
            recurrent_part = recurrent_kernel.T.dot(hidden_state.T).T

            z = input_part + recurrent_part + bias

            z_i, z_f, z_c, z_o = np.split(z, 4, axis=1)

            input_gate = sigmoid(z_i)
            forget_gate = sigmoid(z_f)
            cell_candidate = np.tanh(z_c)
            output_gate = sigmoid(z_o)

            cell_state = (
                forget_gate * cell_state
                + input_gate * cell_candidate
            )

            hidden_state = output_gate * np.tanh(cell_state)

            sequence_output[:, t, :] = hidden_state

        if return_sequences:
            layer_output = sequence_output
        else:
            layer_output = hidden_state

    for layer in dense_layers:

        layer_output = (
            layer_output @ layer["kernel"]
            + layer["bias"]
        )

        if layer["activation"] == "relu":
            layer_output = relu(layer_output)

        elif layer["activation"] == "sigmoid":
            layer_output = sigmoid(layer_output)

    return layer_output

In [13]:
X_validation = X_test_lstm[:10]

keras_output = model.predict(
    X_validation,
    verbose=0
).reshape(-1)

dense_output = manual_lstm_dense(
    X_validation,
    dense_lstm_layers,
    dense_layers
).reshape(-1)

csr_output = manual_lstm_csr(
    X_validation,
    csr_lstm_layers,
    dense_layers
).reshape(-1)

print("Keras output:")
print(keras_output)

print("\nManual dense output:")
print(dense_output)

print("\nManual CSR output:")
print(csr_output)

print("\nMaximum absolute difference:")
print(
    "Keras vs Dense:",
    np.max(np.abs(keras_output - dense_output))
)

print(
    "Keras vs CSR:",
    np.max(np.abs(keras_output - csr_output))
)

print(
    "Dense vs CSR:",
    np.max(np.abs(dense_output - csr_output))
)

Keras output:
[3.5526119e-07 4.6638331e-07 9.9999726e-01 1.0000000e+00 1.0000000e+00
 9.9284577e-01 7.7518689e-07 3.9039257e-03 1.0000000e+00 1.0000000e+00]

Manual dense output:
[3.5526048e-07 4.6638283e-07 9.9999726e-01 1.0000000e+00 1.0000000e+00
 9.9284571e-01 7.7518547e-07 3.9038998e-03 1.0000000e+00 1.0000000e+00]

Manual CSR output:
[3.5526048e-07 4.6638283e-07 9.9999726e-01 1.0000000e+00 1.0000000e+00
 9.9284571e-01 7.7518621e-07 3.9038998e-03 1.0000000e+00 1.0000000e+00]

Maximum absolute difference:
Keras vs Dense: 5.9604645e-08
Keras vs CSR: 5.9604645e-08
Dense vs CSR: 7.3896445e-13


In [14]:
def keras_inference(sample):
    return model.predict(sample, verbose=0)


def dense_inference(sample):
    return manual_lstm_dense(
        sample,
        dense_lstm_layers,
        dense_layers
    )


def csr_inference(sample):
    return manual_lstm_csr(
        sample,
        csr_lstm_layers,
        dense_layers
    )


def measure_inference_time(inference_function, X_timing):
    times = []

    for sample in X_timing:
        sample = np.expand_dims(sample, axis=0)

        start = timeit.default_timer()
        inference_function(sample)
        end = timeit.default_timer()

        times.append((end - start) * 1000)

        gc.collect()

    return np.mean(times), np.std(times)

In [15]:
X_quick_test = X_lstm_timing[:50]

keras_mean, keras_std = measure_inference_time(
    keras_inference,
    X_quick_test
)

dense_mean, dense_std = measure_inference_time(
    dense_inference,
    X_quick_test
)

csr_mean, csr_std = measure_inference_time(
    csr_inference,
    X_quick_test
)

print("=== QUICK BENCHMARK: 50 SAMPLES ===")

print(
    f"Keras:       {keras_mean:.3f} ± {keras_std:.3f} ms"
)

print(
    f"Manual Dense:{dense_mean:.3f} ± {dense_std:.3f} ms"
)

print(
    f"Manual CSR:  {csr_mean:.3f} ± {csr_std:.3f} ms"
)

=== QUICK BENCHMARK: 50 SAMPLES ===
Keras:       46.644 ± 27.138 ms
Manual Dense:6.558 ± 0.320 ms
Manual CSR:  14.037 ± 0.413 ms


In [16]:
keras_mean, keras_std = measure_inference_time(
    keras_inference,
    X_lstm_timing
)

dense_mean, dense_std = measure_inference_time(
    dense_inference,
    X_lstm_timing
)

csr_mean, csr_std = measure_inference_time(
    csr_inference,
    X_lstm_timing
)

print("=== LSTM WEIGHT PRUNED - 10% TEST SET ===")
print(f"Samples: {len(X_lstm_timing)}")
print()

print(
    f"Keras:        {keras_mean:.3f} ± {keras_std:.3f} ms"
)

print(
    f"Manual Dense: {dense_mean:.3f} ± {dense_std:.3f} ms"
)

print(
    f"Manual CSR:   {csr_mean:.3f} ± {csr_std:.3f} ms"
)

print()
print(
    f"CSR / Dense ratio: {csr_mean / dense_mean:.2f}x"
)

=== LSTM WEIGHT PRUNED - 10% TEST SET ===
Samples: 972

Keras:        41.668 ± 1.817 ms
Manual Dense: 6.484 ± 0.388 ms
Manual CSR:   14.059 ± 0.630 ms

CSR / Dense ratio: 2.17x


In [17]:
def paper_style_lstm_csr(model, input_data):
    batch_size = input_data.shape[0]
    layer_outputs = input_data

    for layer in model.layers:

        if isinstance(layer, tf.keras.layers.LSTM):

            kernel, recurrent_kernel, bias = layer.get_weights()

            # Split weights into the four LSTM gates
            kernel_i, kernel_f, kernel_c, kernel_o = np.split(
                kernel, 4, axis=1
            )

            recurrent_i, recurrent_f, recurrent_c, recurrent_o = np.split(
                recurrent_kernel, 4, axis=1
            )

            bias_i, bias_f, bias_c, bias_o = np.split(
                bias, 4
            )

            # Convert weights to CSR inside the inference function
            kernel_i = csr_matrix(kernel_i)
            kernel_f = csr_matrix(kernel_f)
            kernel_c = csr_matrix(kernel_c)
            kernel_o = csr_matrix(kernel_o)

            recurrent_i = csr_matrix(recurrent_i)
            recurrent_f = csr_matrix(recurrent_f)
            recurrent_c = csr_matrix(recurrent_c)
            recurrent_o = csr_matrix(recurrent_o)

            units = layer.units

            # Initialize hidden and cell states as CSR
            hidden_state = csr_matrix(
                np.zeros((batch_size, units), dtype=np.float32)
            )

            cell_state = csr_matrix(
                np.zeros((batch_size, units), dtype=np.float32)
            )

            timesteps = layer_outputs.shape[1]

            sequence_output = np.zeros(
                (batch_size, timesteps, units),
                dtype=np.float32
            )

            for t in range(timesteps):

                # Convert current input to CSR
                current_input = csr_matrix(
                    layer_outputs[:, t, :]
                )

                # Calculate each gate separately
                input_gate = (
                    current_input @ kernel_i
                    + hidden_state @ recurrent_i
                ).toarray()

                input_gate = sigmoid(
                    input_gate + bias_i
                )

                forget_gate = (
                    current_input @ kernel_f
                    + hidden_state @ recurrent_f
                ).toarray()

                forget_gate = sigmoid(
                    forget_gate + bias_f
                )

                cell_gate = (
                    current_input @ kernel_c
                    + hidden_state @ recurrent_c
                ).toarray()

                cell_gate = np.tanh(
                    cell_gate + bias_c
                )

                output_gate = (
                    current_input @ kernel_o
                    + hidden_state @ recurrent_o
                ).toarray()

                output_gate = sigmoid(
                    output_gate + bias_o
                )

                # Convert gates to CSR
                input_gate = csr_matrix(input_gate)
                forget_gate = csr_matrix(forget_gate)
                cell_gate = csr_matrix(cell_gate)
                output_gate = csr_matrix(output_gate)

                # Update cell state
                cell_state = (
                    forget_gate.multiply(cell_state)
                    + input_gate.multiply(cell_gate)
                )

                # Apply tanh to cell state
                cell_activation = csr_matrix(
                    np.tanh(cell_state.toarray())
                )

                # Update hidden state
                hidden_state = output_gate.multiply(
                    cell_activation
                )

                sequence_output[:, t, :] = hidden_state.toarray()

            if layer.return_sequences:
                layer_outputs = sequence_output
            else:
                layer_outputs = hidden_state.toarray()

        elif isinstance(layer, tf.keras.layers.Dense):

            kernel, bias = layer.get_weights()

            # Paper-style conversion of Dense weights
            sparse_kernel = csr_matrix(kernel)

            layer_outputs = csr_matrix(layer_outputs)

            layer_outputs = (
                layer_outputs @ sparse_kernel
            ).toarray()

            layer_outputs = layer_outputs + bias

            activation = layer.activation.__name__

            if activation == "relu":
                layer_outputs = relu(layer_outputs)

            elif activation == "sigmoid":
                layer_outputs = sigmoid(layer_outputs)

            elif activation == "softmax":
                exp_x = np.exp(
                    layer_outputs
                    - np.max(layer_outputs, axis=1, keepdims=True)
                )

                layer_outputs = (
                    exp_x
                    / np.sum(exp_x, axis=1, keepdims=True)
                )

    return np.asarray(layer_outputs)

In [18]:
X_validation = X_test_lstm[:10]

keras_output = model.predict(
    X_validation,
    verbose=0
).reshape(-1)

paper_output = paper_style_lstm_csr(
    model,
    X_validation
).reshape(-1)

print("Keras output:")
print(keras_output)

print("\nPaper-style CSR output:")
print(paper_output)

print("\nMaximum absolute difference:")

difference = np.max(
    np.abs(keras_output - paper_output)
)

print(
    f"Keras vs Paper-style CSR: {difference}"
)

Keras output:
[3.5526119e-07 4.6638331e-07 9.9999726e-01 1.0000000e+00 1.0000000e+00
 9.9284577e-01 7.7518689e-07 3.9039257e-03 1.0000000e+00 1.0000000e+00]

Paper-style CSR output:
[3.5526114e-07 4.6638414e-07 9.9999726e-01 1.0000000e+00 1.0000000e+00
 9.9284571e-01 7.7518689e-07 3.9038998e-03 1.0000000e+00 1.0000000e+00]

Maximum absolute difference:
Keras vs Paper-style CSR: 5.960464477539063e-08


In [19]:
def paper_style_inference(sample):
    return paper_style_lstm_csr(
        model,
        sample
    )

In [20]:
X_quick_test = X_lstm_timing[:50]

paper_mean, paper_std = measure_inference_time(
    paper_style_inference,
    X_quick_test
)

print("=== PAPER-STYLE CSR: 50 SAMPLES ===")
print(
    f"Paper-style CSR: {paper_mean:.3f} ± {paper_std:.3f} ms"
)

print()
print("Previous results:")
print("Manual Dense:       6.558 ms")
print("Preconverted CSR:   14.037 ms")

=== PAPER-STYLE CSR: 50 SAMPLES ===
Paper-style CSR: 155.694 ± 3.265 ms

Previous results:
Manual Dense:       6.558 ms
Preconverted CSR:   14.037 ms


In [21]:
paper_mean, paper_std = measure_inference_time(
    paper_style_inference,
    X_lstm_timing
)

print("=== LSTM WEIGHT PRUNED - PAPER-STYLE CSR ===")
print(f"Samples: {len(X_lstm_timing)}")
print(
    f"Paper-style CSR: {paper_mean:.3f} ± {paper_std:.3f} ms"
)

=== LSTM WEIGHT PRUNED - PAPER-STYLE CSR ===
Samples: 972
Paper-style CSR: 156.916 ± 15.735 ms


# MLP

In [22]:
model_path = MODELS_DIR / "MLP_WeightPruned.keras"

model = tf.keras.models.load_model(model_path)

print("Model:", model_path.name)

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        kernel, bias = layer.get_weights()

        sparsity = np.mean(kernel == 0) * 100

        print(
            f"{layer.name}: "
            f"shape={kernel.shape}, "
            f"sparsity={sparsity:.2f}%"
        )

Model: MLP_WeightPruned.keras
dense: shape=(498, 64), sparsity=60.98%
dense_1: shape=(64, 64), sparsity=60.99%
dense_2: shape=(64, 32), sparsity=60.99%
dense_3: shape=(32, 128), sparsity=60.99%
dense_4: shape=(128, 16), sparsity=60.99%
dense_5: shape=(16, 16), sparsity=60.94%
dense_6: shape=(16, 1), sparsity=62.50%


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(store)


In [23]:
def prepare_mlp_weights(model):
    dense_layers = []
    csr_layers = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            kernel, bias = layer.get_weights()

            dense_layers.append({
                "kernel": kernel.astype(np.float32),
                "bias": bias.astype(np.float32),
                "activation": layer.activation.__name__
            })

            csr_layers.append({
                "kernel": csr_matrix(kernel),
                "bias": bias.astype(np.float32),
                "activation": layer.activation.__name__
            })

    return dense_layers, csr_layers


mlp_dense_layers, mlp_csr_layers = prepare_mlp_weights(model)

In [24]:
def apply_activation(x, activation):
    if activation == "relu":
        return np.maximum(x, 0.0)

    if activation == "sigmoid":
        return 1.0 / (1.0 + np.exp(-np.clip(x, -88.0, 88.0)))

    if activation == "softmax":
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    return x


def manual_mlp_dense(input_data, layers):
    x = np.asarray(input_data, dtype=np.float32)

    if x.ndim > 2:
        x = x.reshape(x.shape[0], -1)

    for layer in layers:
        x = x @ layer["kernel"] + layer["bias"]
        x = apply_activation(x, layer["activation"])

    return x


def manual_mlp_csr(input_data, layers):
    x = np.asarray(input_data, dtype=np.float32)

    if x.ndim > 2:
        x = x.reshape(x.shape[0], -1)

    for layer in layers:
        x = layer["kernel"].T.dot(x.T).T + layer["bias"]
        x = apply_activation(x, layer["activation"])

    return x

In [26]:
model_path = MODELS_DIR / "MLP_WeightPruned.keras"

model = tf.keras.models.load_model(model_path)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model loaded and recompiled.")

Model loaded and recompiled.


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam_1', because it has 30 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(store)


In [27]:
X_validation = X_test[:10]

keras_output = model.predict(
    X_validation,
    verbose=0
).reshape(-1)

dense_output = manual_mlp_dense(
    X_validation,
    mlp_dense_layers
).reshape(-1)

csr_output = manual_mlp_csr(
    X_validation,
    mlp_csr_layers
).reshape(-1)

print("Maximum absolute difference:")
print(
    "Keras vs Dense:",
    np.max(np.abs(keras_output - dense_output))
)
print(
    "Keras vs CSR:",
    np.max(np.abs(keras_output - csr_output))
)
print(
    "Dense vs CSR:",
    np.max(np.abs(dense_output - csr_output))
)

Maximum absolute difference:
Keras vs Dense: 6.556511e-07
Keras vs CSR: 5.9604645e-07
Dense vs CSR: 5.9604645e-08


In [28]:
X_mlp_timing = X_test[timing_idx]


def keras_mlp_inference(sample):
    return model.predict(sample, verbose=0)


def dense_mlp_inference(sample):
    return manual_mlp_dense(
        sample,
        mlp_dense_layers
    )


def csr_mlp_inference(sample):
    return manual_mlp_csr(
        sample,
        mlp_csr_layers
    )

In [29]:
X_quick_test = X_mlp_timing[:50]

keras_mean, keras_std = measure_inference_time(
    keras_mlp_inference,
    X_quick_test
)

dense_mean, dense_std = measure_inference_time(
    dense_mlp_inference,
    X_quick_test
)

csr_mean, csr_std = measure_inference_time(
    csr_mlp_inference,
    X_quick_test
)

print("=== MLP WEIGHT PRUNED - QUICK TEST ===")
print(f"Keras:        {keras_mean:.3f} ± {keras_std:.3f} ms")
print(f"Manual Dense: {dense_mean:.3f} ± {dense_std:.3f} ms")
print(f"Manual CSR:   {csr_mean:.3f} ± {csr_std:.3f} ms")
print(f"CSR / Dense:  {csr_mean / dense_mean:.2f}x")

=== MLP WEIGHT PRUNED - QUICK TEST ===
Keras:        38.503 ± 4.568 ms
Manual Dense: 0.098 ± 0.031 ms
Manual CSR:   0.274 ± 0.033 ms
CSR / Dense:  2.78x


In [ ]:
keras_mean, keras_std = measure_inference_time(
    keras_mlp_inference,
    X_mlp_timing
)

dense_mean, dense_std = measure_inference_time(
    dense_mlp_inference,
    X_mlp_timing
)

csr_mean, csr_std = measure_inference_time(
    csr_mlp_inference,
    X_mlp_timing
)

print("=== MLP WEIGHT PRUNED - 10% TEST SET ===")
print(f"Samples: {len(X_mlp_timing)}")
print()
print(f"Keras:        {keras_mean:.3f} ± {keras_std:.3f} ms")
print(f"Manual Dense: {dense_mean:.3f} ± {dense_std:.3f} ms")
print(f"Manual CSR:   {csr_mean:.3f} ± {csr_std:.3f} ms")
print()
print(f"CSR / Dense ratio: {csr_mean / dense_mean:.2f}x")

In [30]:
def paper_style_mlp_csr(model, input_data):
    layer_outputs = np.asarray(input_data, dtype=np.float32)

    if layer_outputs.ndim > 2:
        layer_outputs = layer_outputs.reshape(layer_outputs.shape[0], -1)

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):

            weights, bias = layer.get_weights()

            # Convert weights to CSR inside each inference
            sparse_weights = csr_matrix(weights)

            # Convert current activations to CSR
            sparse_input = csr_matrix(layer_outputs)

            # Sparse matrix multiplication
            layer_outputs = (
                sparse_input @ sparse_weights
            ).toarray()

            layer_outputs = layer_outputs + bias

            activation = layer.activation.__name__

            if activation == "relu":
                layer_outputs = np.maximum(layer_outputs, 0.0)

            elif activation == "sigmoid":
                layer_outputs = 1.0 / (
                    1.0 + np.exp(
                        -np.clip(layer_outputs, -88.0, 88.0)
                    )
                )

            elif activation == "softmax":
                exp_x = np.exp(
                    layer_outputs
                    - np.max(layer_outputs, axis=1, keepdims=True)
                )

                layer_outputs = (
                    exp_x
                    / np.sum(exp_x, axis=1, keepdims=True)
                )

    return layer_outputs

In [31]:
X_validation = X_test[:10]

keras_output = model.predict(
    X_validation,
    verbose=0
).reshape(-1)

paper_output = paper_style_mlp_csr(
    model,
    X_validation
).reshape(-1)

difference = np.max(
    np.abs(keras_output - paper_output)
)

print("Maximum absolute difference:")
print(f"Keras vs Paper-style CSR: {difference}")

Maximum absolute difference:
Keras vs Paper-style CSR: 5.960464477539062e-07


In [32]:
def paper_style_mlp_inference(sample):
    return paper_style_mlp_csr(
        model,
        sample
    )


X_quick_test = X_mlp_timing[:50]

paper_mean, paper_std = measure_inference_time(
    paper_style_mlp_inference,
    X_quick_test
)

print("=== MLP WEIGHT PRUNED - PAPER-STYLE CSR ===")
print(f"Paper-style CSR: {paper_mean:.3f} ± {paper_std:.3f} ms")

print()
print("Previous results:")
print("Keras:             38.503 ms")
print("Manual Dense:       0.098 ms")
print("Preconverted CSR:   0.274 ms")

=== MLP WEIGHT PRUNED - PAPER-STYLE CSR ===
Paper-style CSR: 2.799 ± 0.182 ms

Previous results:
Keras:             38.503 ms
Manual Dense:       0.098 ms
Preconverted CSR:   0.274 ms


# LSTMS Pruning

In [33]:
# Preconverted CSR benchmark for all weight-pruned LSTM models

lstm_models = {
    "LSTM_Weight": "LSTM_WeightPruned.keras",
    "LSTM_WeightNode": "LSTM_WeightNodePruned.keras",
    "LSTM_NodeWeight": "LSTM_NodeWeightPruned.keras"
}

csr_results = []

for model_name, filename in lstm_models.items():

    model_path = MODELS_DIR / filename
    model = tf.keras.models.load_model(model_path)

    # Convert LSTM weights to CSR once before timing
    _, csr_lstm_layers, dense_layers = prepare_lstm_weights(model)

    # Calculate LSTM weight sparsity
    total_weights = 0
    total_zeros = 0

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.LSTM):
            kernel, recurrent_kernel, _ = layer.get_weights()

            total_weights += kernel.size + recurrent_kernel.size
            total_zeros += (
                np.count_nonzero(kernel == 0)
                + np.count_nonzero(recurrent_kernel == 0)
            )

    sparsity = (total_zeros / total_weights) * 100

    # Inference function using already prepared CSR weights
    def inference(sample):
        return manual_lstm_csr(
            sample,
            csr_lstm_layers,
            dense_layers
        )

    mean_time, std_time = measure_inference_time(
        inference,
        X_lstm_timing
    )

    csr_results.append({
        "Model": model_name,
        "Sparsity (%)": sparsity,
        "Samples": len(X_lstm_timing),
        "CSR Mean (ms)": mean_time,
        "CSR Std (ms)": std_time
    })

    print(
        f"{model_name}: "
        f"{mean_time:.3f} ± {std_time:.3f} ms"
    )

    del model
    gc.collect()


csr_lstm_df = pd.DataFrame(csr_results)

print("\n=== PRECONVERTED CSR - LSTM ===")
display(csr_lstm_df)

csr_lstm_df.to_csv(
    RESULTS_DIR / "LSTM_preconverted_CSR_results.csv",
    index=False
)

LSTM_Weight: 14.105 ± 0.868 ms
LSTM_WeightNode: 13.840 ± 0.501 ms
LSTM_NodeWeight: 13.738 ± 0.492 ms

=== PRECONVERTED CSR - LSTM ===


,Model,Sparsity (%),Samples,CSR Mean (ms),CSR Std (ms)
0,LSTM_Weight,64.990668,972,14.104705,0.868080
1,LSTM_WeightNode,64.434464,972,13.840127,0.501486
2,LSTM_NodeWeight,63.497834,972,13.738181,0.491711


In [34]:
# Preconverted CSR benchmark for all weight-pruned MLP models

mlp_models = {
    "MLP_Weight": "MLP_WeightPruned.keras",
    "MLP_WeightNode": "MLP_WeightNodePruned.keras",
    "MLP_NodeWeight": "MLP_NodeWeightPruned.keras"
}

csr_results = []

for model_name, filename in mlp_models.items():

    model_path = MODELS_DIR / filename
    model = tf.keras.models.load_model(model_path)

    # Convert weights to CSR once before timing
    _, csr_layers = prepare_mlp_weights(model)

    # Calculate total Dense weight sparsity
    total_weights = 0
    total_zeros = 0

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            kernel, _ = layer.get_weights()

            total_weights += kernel.size
            total_zeros += np.count_nonzero(kernel == 0)

    sparsity = (total_zeros / total_weights) * 100

    # Inference using preconverted CSR weights
    def inference(sample):
        return manual_mlp_csr(
            sample,
            csr_layers
        )

    mean_time, std_time = measure_inference_time(
        inference,
        X_mlp_timing
    )

    csr_results.append({
        "Model": model_name,
        "Sparsity (%)": sparsity,
        "Samples": len(X_mlp_timing),
        "CSR Mean (ms)": mean_time,
        "CSR Std (ms)": std_time
    })

    print(
        f"{model_name}: "
        f"{mean_time:.3f} ± {std_time:.3f} ms"
    )

    del model
    gc.collect()


csr_mlp_df = pd.DataFrame(csr_results)

print("\n=== PRECONVERTED CSR - MLP ===")
display(csr_mlp_df)

csr_mlp_df.to_csv(
    RESULTS_DIR / "MLP_preconverted_CSR_results.csv",
    index=False
)

c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam_3', because it has 30 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(store)


MLP_Weight: 0.277 ± 0.045 ms
MLP_WeightNode: 0.268 ± 0.040 ms


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


MLP_NodeWeight: 0.267 ± 0.047 ms

=== PRECONVERTED CSR - MLP ===


,Model,Sparsity (%),Samples,CSR Mean (ms),CSR Std (ms)
0,MLP_Weight,60.980825,972,0.277168,0.045433
1,MLP_WeightNode,58.033479,972,0.268459,0.039603
2,MLP_NodeWeight,60.976460,972,0.267085,0.046592
